# Models

A key component of `synthpop` is the concept of a **model**. Models describe the physical properties of a galaxy population and define the assumptions used to generate synthetic galaxies.

For stellar populations, a model typically specifies:

- The stellar mass function
- The star formation history
- The chemical enrichment (metallicity) history
- Dust attenuation

Both `synthpop` and `synthesizer` provide the machinery required to construct and work with these models. Together, they enable users to generate physically motivated galaxy populations and predict their observable properties.

`synthpop` also includes a variety of pre-made models.

In [ ]:
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from synthesizer.parametric import SFH, Stars, ZDist
from synthpop.distribution_functions import Schechter
from synthpop.models import Model, Spheroid, Default, Constant
from unyt import yr, Myr, Msun, Gyr, unyt_quantity, Mpc, dimensionless


## Galaxy Stellar Mass Function

The first component that is needed for `synthpop` 

In [ ]:

# Define the galaxy stellar mass function (GSMF) using a Schechter function
x_star = 10**10.745 * Msun     
phi_star = 10**(-2.437) * Mpc**-3  
alpha = -1.465
gsmf = Schechter(x_star=x_star, alpha=alpha, phi_star=phi_star)

## Star formation and metal enrichment history

In [ ]:

# Define a delta function for metallicity
metal_dist_function = ZDist.DeltaConstant

metal_dist_parameters = {
    "log10metallicity": -2.5
}


age_of_universe = unyt_quantity(cosmo.age(0).value, str(cosmo.age(0).unit))

sfh_function = SFH.LogNormal

def peak_age_function(mass, age_of_Universe=1.37E10 * yr):

    value = (mass/(1E9*Msun))**2.5 * 1E9 + np.random.normal(0, 1e9)

    return np.min((value, age_of_Universe.to('yr').value)) * yr

def tau_function(mass):
    return np.clip(np.random.normal(0.6, 0.1) * dimensionless, 0.1, 1.0)

sfh_parameters = {
    "tau": 0.6 * dimensionless,
    "peak_age": 1e10 * yr,
    "max_age": age_of_universe
}

# sfh_parameters = {
#     "tau": tau_function,
#     "peak_age": peak_age_function,
#     "max_age": age_of_universe
# }


sfh_function = SFH.Constant

sfh_parameters = {
    "max_age": 10000 * Myr
}



In [ ]:
plt.plot(np.linspace(0, 10000, 100), sfh_function(**sfh_parameters).get_sfr(np.linspace(0, 10000, 100)*Myr))
plt.show()

## Dust attenuation

Our model can also incorporate a description of dust attenuation, parameterised by the $V$-band optical depth $\tau_V$, affecting the stellar population. This parameter could be taken as constant; however, a more realistic treatment would allow the dust attenuation to increase with stellar mass, with additional scatter about the mean relation.

In the simple model implemented here, the dust attenuation is assumed to increase linearly with stellar mass above $10^{9},M_{\odot}$ and to be zero below this threshold. Random scatter is also included, with an amplitude proportional to the characteristic value of $\tau_V$.

In [ ]:
# Define dust attenuation as a function of stellar mass
def dust_attenuation_function(mass):

    tau_v = 0.1 * ((mass-1E9*Msun)/(1E9*Msun))  # Example: tau_v decreases with mass
    r = np.random.normal(0, 0.5, size=mass.shape)  # Add some scatter
    return np.maximum(0.0, tau_v * (1 + r)) * dimensionless


log_masses = np.random.uniform(8, 11, size=100)
masses = (10 ** log_masses) * Msun
tau_v_values = dust_attenuation_function(masses)

plt.scatter(masses/Msun, tau_v_values)
plt.xscale('log')
plt.xlabel('Stellar Mass (Msun)')
plt.ylabel('Dust Attenuation (tau_v)')
plt.title('Dust Attenuation vs Stellar Mass')
plt.show()

## Creating a Model

For convenience, we can combine all of these model components into an instance of a `Model` class, which can then be passed to our galaxy population generator. The `Model` class also provides additional utility methods for visualising and inspecting the model.

In [ ]:
# Define the Model with the specified components
model = Model(        
    galaxy_stellar_mass_function=gsmf,
    sfh_function=sfh_function,
    sfh_parameters=sfh_parameters,
    metal_dist_function=metal_dist_function,
    metal_dist_parameters=metal_dist_parameters,
    dust_attenuation_function=dust_attenuation_function,
)

# Visualise the SFH for a galaxy of mass 1e10 Msun over the age of the universe
ages = np.linspace(0, 10000, 100) * Myr
mass = 1e10 * Msun
model.plot_sfh(mass, ages)


## Pre-built models

`Synthpop` contains a number of pre-built models, which combine a galaxy stellar mass function (GSMF), or sub-halo abundance matching (SHAM) model with a model which describes the star formation and metal enrichment history and dust attenuation. 

In [ ]:
ages = np.linspace(0, 14000, 100) * Myr
mass = 1e10 * Msun

for model in [Spheroid(), Default(), Constant()]:
    model.plot_sfh(mass, ages)